# Progetto 5G Networks
## Testo da trasmettere:

In [6]:
text = """A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
   THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.   
The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual entities. These semantic aspects of communication are irrelevant to the engineering problem. The significant aspect is that the actual message is one selected from a set of possible messages. The system must be designed to operate for each possible selection, not just the one which will actually be chosen since this is unknown at the time of design.   
If the number of messages in the set is finite then this number or any monotonic function of this number can be regarded as a measure of the information produced when one message is chosen from the set, all choices being equally likely. As was pointed out by Hartley the most natural choice is the logarithmic function. Although this definition must be generalized considerably when we consider the influence of the statistics of the message and when we have a continuous range of messages, we will in all cases use an essentially logarithmic measure. """
print(text)


A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
   THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.   
The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual entities. These semantic aspects of communication are irrelev

## Codifica

In [7]:
import numpy as np
bits = np.unpackbits(np.frombuffer(text.encode('utf-8'), dtype=np.uint8))
print(len(bits))

14768


In [8]:
def lz78_compress(bits):
    dizionario = {"": 0}      # frase -> indice
    out = []                  # lista di coppie (indice, bit)
    w = ""                    # frase corrente

    for b in bits:
        c = str(int(b))
        if w + c in dizionario:
            w += c
        else:
            out.append((dizionario[w], int(c)))
            dizionario[w + c] = len(dizionario)
            w = ""

    if w:  # l'input è finito su una frase già nel dizionario
        out.append((dizionario[w], None))
    return out


def lz78_decompress(coppie):
    frasi = [""]
    out = []
    for idx, c in coppie:
        if c is None:                  # coppia finale senza bit nuovo
            out.append(frasi[idx])
        else:
            frase = frasi[idx] + str(c)
            out.append(frase)
            frasi.append(frase)
    s = "".join(out)
    return np.array(list(s), dtype=np.uint8)

In [9]:
compresso = lz78_compress(bits)
print(len(compresso))

1469


In [10]:
decompresso = lz78_decompress(compresso)
print(len(decompresso))

14768


In [11]:
text_back = np.packbits(bits).tobytes().decode('utf-8')
print(text_back)

A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
   THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.   
The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual entities. These semantic aspects of communication are irrelev

In [12]:
print(text == text_back)

True


In [27]:
import math
import numpy as np

def lz78_compress_chars(text):
    dizionario = {"": 0}
    out = []
    w = ""
    for c in text:
        if w + c in dizionario:
            w += c
        else:
            out.append((dizionario[w], c))
            dizionario[w + c] = len(dizionario)
            w = ""
    if w:
        out.append((dizionario[w], None))
    return out


def lz78_decompress_chars(coppie):
    frasi = [""]
    out = []
    for idx, c in coppie:
        if c is None:
            out.append(frasi[idx])
        else:
            frase = frasi[idx] + c
            out.append(frase)
            frasi.append(frase)
    return "".join(out)

def _bit_char(alfabeto):
    # +1 per il valore riservato alla coppia finale senza simbolo (None)
    return max(1, math.ceil(math.log2(len(alfabeto) + 1)))


def lz78_to_bits(coppie, alfabeto):
    pos = {c: i for i, c in enumerate(alfabeto)}
    bc = _bit_char(alfabeto)
    out = []
    for k, (idx, c) in enumerate(coppie, start=1):
        n_idx = max(1, math.ceil(math.log2(k)))
        out.append(format(idx, f'0{n_idx}b'))
        val = pos[c] if c is not None else len(alfabeto)
        out.append(format(val, f'0{bc}b'))
    return np.array(list(''.join(out)), dtype=np.uint8)


def bits_to_lz78(bits, alfabeto):
    bc = _bit_char(alfabeto)
    s = ''.join(str(int(b)) for b in bits)
    coppie = []
    i = 0
    k = 1
    while i < len(s):
        n_idx = max(1, math.ceil(math.log2(k)))
        if i + n_idx + bc > len(s):
            raise ValueError("Flusso di bit troncato o corrotto")
        idx = int(s[i:i + n_idx], 2)
        i += n_idx
        val = int(s[i:i + bc], 2)
        i += bc

        if val == len(alfabeto):          # codice riservato: coppia finale
            c = None
        elif val < len(alfabeto):
            c = alfabeto[val]
        else:
            raise ValueError("Simbolo non valido nel flusso")

        if idx >= k:                      # indice oltre il dizionario corrente
            raise ValueError("Indice non valido nel flusso")

        coppie.append((idx, c))
        k += 1
    return coppie

In [21]:
compresso = lz78_compress_chars(text)
print(len(compresso), "coppie")                       # numero di coppie
assert lz78_decompress_chars(compresso) == text

645 coppie


In [25]:
alfabeto = sorted(set(text))
compresso = lz78_compress_chars(text)
bits_compressi = lz78_to_bits(compresso, alfabeto)
print(len(bits_compressi), "bit")

9298 bit


In [24]:
coppie = bits_to_lz78(bits_compressi, alfabeto);
print(len(coppie), "coppie")

645 coppie


In [29]:
testo_ricevuto = lz78_decompress_chars(coppie)
print(testo_ricevuto)

A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
   THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.   
The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual entities. These semantic aspects of communication are irrelev